# CWE Forest API Demo

This notebook demonstrates all available APIs for the `cwe_tree` module, which provides programmatic access to the Common Weakness Enumeration (CWE) forest structure.

The CWE forest is a hierarchical graph of security weaknesses where:
- **Nodes** represent individual CWE weaknesses
- **Edges** represent parent-child relationships
- **Multiple roots** create an independent tree structure (forest)

## Table of Contents
1. [Installation & Import](#installation--import)
2. [Basic Node Access](#basic-node-access)
3. [Parent-Child Navigation](#parent-child-navigation)
4. [Forest Structure](#forest-structure)
5. [Graph Traversal](#graph-traversal)
6. [Metadata Retrieval](#metadata-retrieval)
7. [Visualization](#visualization)
8. [Advanced Examples](#advanced-examples)

## Installation & Import

In [45]:
# Import the module
# !pip install cwe-tree
from cwe_tree import query

# `query` is a pre-loaded singleton CweForest instance
print(f"CweForest type: {type(query)}")
print(f"CweForest instance: {query}")

CweForest type: <class 'cwe_tree._forest.CweForest'>
CweForest instance: <cwe_tree._forest.CweForest object at 0x10af59fd0>


## Basic Node Access

### `node(cwe_id) -> Optional[CweNode]`

Retrieve a specific CWE node by its ID. The ID is automatically normalized (e.g., "284" becomes "CWE-284").

In [46]:
# Get a node with full ID
node_79 = query.get_cwe("CWE-79")
if node_79:
    print(f"Found: {node_79.cwe_id}")
    print(f"Name: {node_79.name}")
    print(f"Abstract: {node_79.abstract}")
else:
    print("Node not found")

Found: CWE-79
Name: Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')
Abstract: Base


In [47]:
# Get a node with normalized ID (ID normalization works automatically)
node_284 = query.get_cwe("284")  # Automatically becomes "CWE-284"
if node_284:
    print(f"Found: {node_284.cwe_id}")
    print(f"Name: {node_284.name}")
else:
    print("Node not found")

Found: CWE-284
Name: Improper Access Control


In [48]:
# Handle non-existent nodes
non_existent = query.get_cwe("CWE-99999")
print(f"Result for non-existent node: {non_existent}")

if non_existent is None:
    print("Node not found - gracefully handled")

Result for non-existent node: None
Node not found - gracefully handled


## Parent-Child Navigation

### `get_parents(cwe_id) -> Set[CweNode]`

Retrieve all parent nodes of a given CWE weakness.

In [49]:
# Get parents of a node
node = query.get_cwe("CWE-77")

parents = query.get_parents("CWE-77")
print(f"Parents of CWE-77:")
for parent in parents:
    print(f"  - {parent.cwe_id}: {parent.name}")

print(f"\nTotal parents: {len(parents)}")

Parents of CWE-77:
  - CWE-74: Improper Neutralization of Special Elements in Output Used by a Downstream Component ('Injection')

Total parents: 1


### `get_children(cwe_id) -> Set[CweNode]`

Retrieve all child nodes of a given CWE weakness.

In [50]:
# Get children of a node
children = sorted(query.get_children("CWE-77"), key=lambda x: x.cwe_id)
print(f"Children of CWE-77:")
for child in children:  
    print(f"  - {child.cwe_id}: {child.name}")

print(f"\nTotal children: {len(children)}")

Children of CWE-77:
  - CWE-624: Executable Regular Expression Error
  - CWE-78: Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')
  - CWE-88: Improper Neutralization of Argument Delimiters in a Command ('Argument Injection')
  - CWE-917: Improper Neutralization of Special Elements used in an Expression Language Statement ('Expression Language Injection')

Total children: 4


## Forest Structure

### `get_root_nodes() -> list[CweNode]`

Retrieve all root nodes in the CWE forest. Root nodes are nodes with no parents, forming the top level of independent tree hierarchies.

In [51]:
# Get all root nodes
roots = query.get_root_nodes()
print(f"Forest has {len(roots)} root node(s):\n")

for root in roots:
    print(f"Root: {root.cwe_id}")
    print(f"  Name: {root.name}")
    print(f"  Abstract: {root.abstract}")
    print()

Forest has 10 root node(s):

Root: CWE-284
  Name: Improper Access Control
  Abstract: Pillar

Root: CWE-435
  Name: Improper Interaction Between Multiple Correctly-Behaving Entities
  Abstract: Pillar

Root: CWE-664
  Name: Improper Control of a Resource Through its Lifetime
  Abstract: Pillar

Root: CWE-682
  Name: Incorrect Calculation
  Abstract: Pillar

Root: CWE-691
  Name: Insufficient Control Flow Management
  Abstract: Pillar

Root: CWE-693
  Name: Protection Mechanism Failure
  Abstract: Pillar

Root: CWE-697
  Name: Incorrect Comparison
  Abstract: Pillar

Root: CWE-703
  Name: Improper Check or Handling of Exceptional Conditions
  Abstract: Pillar

Root: CWE-707
  Name: Improper Neutralization
  Abstract: Pillar

Root: CWE-710
  Name: Improper Adherence to Coding Standards
  Abstract: Pillar



## Graph Traversal

These methods are provided by the underlying `AbcGraphQuerier` base class from cpg2py.

### `succ(node) -> Iterable[CweNode]`

Returns all successor nodes (children) connected via outgoing edges.

In [52]:
# Use succ() directly
node = query.get_cwe("CWE-77")

successors = list(query.succ(node))[:5]  # Get first 5
print(f"First 5 successors of {node.cwe_id}:")
for succ in successors:
    print(f"  - {succ.cwe_id}: {succ.name}")

First 5 successors of CWE-77:
  - CWE-624: Executable Regular Expression Error
  - CWE-78: Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')
  - CWE-88: Improper Neutralization of Argument Delimiters in a Command ('Argument Injection')
  - CWE-917: Improper Neutralization of Special Elements used in an Expression Language Statement ('Expression Language Injection')


### `prev(node) -> Iterable[CweNode]`

Returns all predecessor nodes (parents) connected via incoming edges.

In [53]:
# Use prev() directly
node = query.get_cwe("CWE-77")

predecessors = list(query.prev(node))
print(f"Predecessors of {node.cwe_id}:")
for pred in predecessors:
    print(f"  - {pred.cwe_id}: {pred.name}")

Predecessors of CWE-77:
  - CWE-74: Improper Neutralization of Special Elements in Output Used by a Downstream Component ('Injection')


### `descendants(node, filter=None) -> Iterable[CweNode]`

Performs breadth-first traversal to find all nodes reachable from the source node (all descendants).

In [54]:
# Get all descendants
root = query.get_root_nodes()[0]
all_descendants = list(query.descendants(root))

print(f"Total descendants of {root.cwe_id}: {len(all_descendants)}")
print(f"First 10 descendants:")
for desc in all_descendants[:10]:
    print(f"  - {desc.cwe_id}: {desc.name}")

Total descendants of CWE-284: 148
First 10 descendants:
  - CWE-1191: On-Chip Debug and Test Interface With Improper Access Control
  - CWE-1220: Insufficient Granularity of Access Control
  - CWE-1224: Improper Restriction of Write-Once Bit Fields
  - CWE-1231: Improper Prevention of Lock Bit Modification
  - CWE-1233: Security-Sensitive Hardware Controls with Missing Lock Bit Protection
  - CWE-1242: Inclusion of Undocumented Features or Chicken Bits
  - CWE-1252: CPU Hardware Not Configured to Support Exclusivity of Write and Execute Operations
  - CWE-1257: Improper Access Control Applied to Mirrored or Aliased Memory Regions
  - CWE-1259: Improper Restriction of Security Token Assignment
  - CWE-1260: Improper Handling of Overlap Between Protected Memory Ranges


### `ancestors(node, filter=None) -> Iterable[CweNode]`

Performs breadth-first traversal to find all nodes from which the source node is reachable (all ancestors).

In [55]:
# Get all ancestors
node = query.get_cwe("CWE-77")
all_ancestors = list(query.ancestors(node))

print(f"All ancestors of {node.cwe_id}: {len(all_ancestors)}")
for anc in all_ancestors:
    print(f"  - {anc.cwe_id}: {anc.name}")

All ancestors of CWE-77: 2
  - CWE-74: Improper Neutralization of Special Elements in Output Used by a Downstream Component ('Injection')
  - CWE-707: Improper Neutralization


### `nodes(predicate=None) -> Iterable[CweNode]`

Iterate over all nodes in the forest, optionally filtered by a predicate function.

In [56]:
# Get all nodes
all_nodes = list(query.nodes())
print(f"Total nodes in forest: {len(all_nodes)}")

Total nodes in forest: 938


In [57]:
# Filter nodes by predicate
class_nodes = list(query.nodes(lambda n: n.abstract == "Class"))
print(f"Total 'Class' abstract type nodes: {len(class_nodes)}")
print(f"First 5 Class nodes:")
for node in class_nodes[:5]:
    print(f"  - {node.cwe_id}: {node.name}")

Total 'Class' abstract type nodes: 110
First 5 Class nodes:
  - CWE-732: Incorrect Permission Assignment for Critical Resource
  - CWE-285: Improper Authorization
  - CWE-1263: Improper Physical Access Control
  - CWE-863: Incorrect Authorization
  - CWE-923: Improper Restriction of Communication Channel to Intended Endpoints


### `edges(predicate=None) -> Iterable[CweEdge]`

Iterate over all edges in the forest, optionally filtered by a predicate function.

In [58]:
# Get all edges
all_edges = list(query.edges())
print(f"Total edges in forest: {len(all_edges)}")

# Show first 5 edges
print(f"\nFirst 5 edges:")
for edge in all_edges[:5]:
    print(f"  {edge.from_nid} -> {edge.to_nid} ({edge.edge_type})")

Total edges in forest: 928

First 5 edges:
  CWE-732 -> CWE-1004 (CHILD)
  CWE-732 -> CWE-276 (CHILD)
  CWE-732 -> CWE-277 (CHILD)
  CWE-732 -> CWE-278 (CHILD)
  CWE-732 -> CWE-279 (CHILD)


### `first_node(predicate=None) -> Optional[CweNode]`

Returns the first node matching the predicate, or None if no match found.

In [59]:
# Find first node with specific property
first_base = query.first_node(lambda n: n.abstract == "Base")
if first_base:
    print(f"First Base node found: {first_base.cwe_id}")
    print(f"  Name: {first_base.name}")

First Base node found: CWE-266
  Name: Incorrect Privilege Assignment


## Metadata Retrieval

### `get_metadata(cwe_id) -> Optional[dict]`

Retrieve complete metadata for a node including its relationships.

In [60]:
# Get comprehensive metadata
import json

metadata = query.get_metadata("CWE-79")
print(json.dumps(metadata, indent=2))

{
  "id": "CWE-79",
  "name": "Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')",
  "abstract": "Base",
  "layer": {
    "CWE-707": 2
  },
  "parents": [
    "CWE-74"
  ],
  "children": [
    "CWE-83",
    "CWE-81",
    "CWE-85",
    "CWE-80",
    "CWE-84",
    "CWE-87",
    "CWE-86"
  ]
}


### `get_layer(cwe_id) -> dict`

Retrieve layer information showing the depth of a node in different root hierarchies.

In [61]:
# Get layer information
layer = query.get_layer("CWE-77")
print(f"Layer information for CWE-77:")
print(json.dumps(layer, indent=2))

Layer information for CWE-77:
{
  "CWE-707": 2
}


## Visualization

### `show(cwe_id=None) -> None`

Display the forest structure with tree-like ASCII formatting.

In [62]:
# Display entire forest from root nodes
print("Displaying entire CWE forest structure:")
print("="*60)
query.show()

Displaying entire CWE forest structure:
CWE-284: Improper Access Control
├── CWE-749: Exposed Dangerous Method or Function
│   ├── CWE-782: Exposed IOCTL with Insufficient Access Control
│   ├── CWE-618: Exposed Unsafe ActiveX Method
├── CWE-1283: Mutable Attestation or Measurement Reporting Data
├── CWE-1224: Improper Restriction of Write-Once Bit Fields
├── CWE-1290: Incorrect Decoding of Security Identifiers 
├── CWE-1294: Insecure Security Identifier Mechanism
│   ├── CWE-1302: Missing Source Identifier in Entity Transactions on a System-On-Chip (SOC)
├── CWE-1259: Improper Restriction of Security Token Assignment
├── CWE-1292: Incorrect Conversion of Security Identifiers
├── CWE-1260: Improper Handling of Overlap Between Protected Memory Ranges
├── CWE-1296: Incorrect Chaining or Granularity of Debug Components
├── CWE-1257: Improper Access Control Applied to Mirrored or Aliased Memory Regions
├── CWE-1231: Improper Prevention of Lock Bit Modification
├── CWE-1304: Improperly Pres

In [63]:
# Display specific subtree
print("Displaying subtree from CWE-77:")
print("="*60)
query.show("CWE-77")

Displaying subtree from CWE-77:
CWE-77: Improper Neutralization of Special Elements used in a Command ('Command Injection')
├── CWE-917: Improper Neutralization of Special Elements used in an Expression Language Statement ('Expression Language Injection')
├── CWE-88: Improper Neutralization of Argument Delimiters in a Command ('Argument Injection')
├── CWE-624: Executable Regular Expression Error
├── CWE-78: Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')


In [64]:
# Display specific subtree (with normalization)
print("Displaying subtree from node 284:")
print("="*60)
query.show("284")  # Automatically normalized to CWE-284

Displaying subtree from node 284:
CWE-284: Improper Access Control
├── CWE-749: Exposed Dangerous Method or Function
│   ├── CWE-618: Exposed Unsafe ActiveX Method
│   ├── CWE-782: Exposed IOCTL with Insufficient Access Control
├── CWE-287: Improper Authentication
│   ├── CWE-306: Missing Authentication for Critical Function
│   │   ├── CWE-288: Authentication Bypass Using an Alternate Path or Channel
│   │   ├── CWE-322: Key Exchange without Entity Authentication
│   ├── CWE-295: Improper Certificate Validation
│   │   ├── CWE-299: Improper Check for Certificate Revocation
│   │   │   ├── CWE-370: Missing Check for Certificate Revocation after Initial Check
│   │   ├── CWE-298: Improper Validation of Certificate Expiration
│   │   ├── CWE-296: Improper Following of a Certificate's Chain of Trust
│   │   ├── CWE-599: Missing Validation of OpenSSL Certificate
│   ├── CWE-645: Overly Restrictive Account Lockout Mechanism
│   ├── CWE-1390: Weak Authentication
│   │   ├── CWE-263: Password

## Advanced Examples

Practical use cases combining multiple APIs.

### Example 1: Find all Variant-type weaknesses

In [65]:
# Find all Variant type nodes
variant_nodes = list(query.nodes(lambda n: n.abstract == "Variant"))
print(f"Total Variant nodes: {len(variant_nodes)}")
print(f"\nFirst 10 Variant nodes:")
for node in variant_nodes[:10]:
    print(f"  {node.cwe_id}: {node.name}")

Total Variant nodes: 292

First 10 Variant nodes:
  CWE-1004: Sensitive Cookie Without 'HttpOnly' Flag
  CWE-1022: Use of Web Link to Untrusted Target with window.opener Access
  CWE-1222: Insufficient Granularity of Address Regions Protected by Register Locks
  CWE-1275: Sensitive Cookie with Improper SameSite Attribute
  CWE-13: ASP.NET Misconfiguration: Password in Configuration File
  CWE-258: Empty Password in Configuration File
  CWE-259: Use of Hard-coded Password
  CWE-277: Insecure Inherited Permissions
  CWE-278: Insecure Preserved Inherited Permissions
  CWE-279: Incorrect Execution-Assigned Permissions


### Example 2: Analyze a specific weakness hierarchy

In [66]:
# Analyze CWE-79 (XSS) hierarchy
cwe_79 = query.get_cwe("CWE-79")

print(f"=== CWE-79 (XSS) Analysis ===")
print(f"Name: {cwe_79.name}")
print(f"Abstract: {cwe_79.abstract}")

# Get parents
parents = query.get_parents("CWE-79")
print(f"\nParents ({len(parents)}):")
for parent in parents:
    print(f"  - {parent.cwe_id}: {parent.name}")

# Get children
children = query.get_children("CWE-79")
print(f"\nChildren ({len(children)}):")
for child in list(children)[:5]:
    print(f"  - {child.cwe_id}: {child.name}")
if len(children) > 5:
    print(f"  ... and {len(children) - 5} more")

# Get all descendants
descendants = list(query.descendants(cwe_79))
print(f"\nAll descendants: {len(descendants)}")

=== CWE-79 (XSS) Analysis ===
Name: Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')
Abstract: Base

Parents (1):
  - CWE-74: Improper Neutralization of Special Elements in Output Used by a Downstream Component ('Injection')

Children (7):
  - CWE-84: Improper Neutralization of Encoded URI Schemes in a Web Page
  - CWE-85: Doubled Character XSS Manipulations
  - CWE-87: Improper Neutralization of Alternate XSS Syntax
  - CWE-86: Improper Neutralization of Invalid Characters in Identifiers in Web Pages
  - CWE-83: Improper Neutralization of Script in Attributes in a Web Page
  ... and 2 more

All descendants: 8


### Example 3: Find nodes by name pattern

In [67]:
# Find all nodes with "Buffer" in the name
buffer_nodes = list(query.nodes(lambda n: "Buffer" in n.name))
print(f"Nodes with 'Buffer' in name: {len(buffer_nodes)}")
print()
for node in buffer_nodes:
    print(f"  {node.cwe_id}: {node.name}")

Nodes with 'Buffer' in name: 16

  CWE-14: Compiler Removal of Code to Clear Buffers
  CWE-119: Improper Restriction of Operations within the Bounds of a Memory Buffer
  CWE-120: Buffer Copy without Checking Size of Input ('Classic Buffer Overflow')
  CWE-788: Access of Memory Location After End of Buffer
  CWE-121: Stack-based Buffer Overflow
  CWE-122: Heap-based Buffer Overflow
  CWE-786: Access of Memory Location Before Start of Buffer
  CWE-124: Buffer Underwrite ('Buffer Underflow')
  CWE-126: Buffer Over-read
  CWE-127: Buffer Under-read
  CWE-761: Free of Pointer not at Start of Buffer
  CWE-805: Buffer Access with Incorrect Length Value
  CWE-806: Buffer Access Using Size of Source Buffer
  CWE-131: Incorrect Calculation of Buffer Size
  CWE-680: Integer Overflow to Buffer Overflow
  CWE-785: Use of Path Manipulation Function without Maximum-sized Buffer


### Example 4: Trace ancestry path

In [68]:
# Find complete ancestry path from a specific node to root
def get_path_to_root(forest, node_id):
    """Get the path from a node to its root ancestor."""
    path = []
    current = forest.get_cwe(node_id)
    visited = set()
    
    while current and current.cwe_id not in visited:
        path.append(current)
        visited.add(current.cwe_id)
        
        parents = forest.get_parents(current.cwe_id)
        if parents:
            current = list(parents)[0]  # Follow first parent
        else:
            break
    
    return path

# Example: trace path for CWE-284
path = get_path_to_root(query, "CWE-77")
print(f"Path from CWE-77 to root:")
for i, node in enumerate(path):
    indent = "  " * i
    print(f"{indent}-> {node.cwe_id}: {node.name}")

Path from CWE-77 to root:
-> CWE-77: Improper Neutralization of Special Elements used in a Command ('Command Injection')
  -> CWE-74: Improper Neutralization of Special Elements in Output Used by a Downstream Component ('Injection')
    -> CWE-707: Improper Neutralization


### Example 5: Forest Statistics

In [69]:
# Compute forest statistics
all_nodes = list(query.nodes())
all_edges = list(query.edges())
roots = query.get_root_nodes()

# Count by abstract type
abstract_counts = {}
for node in all_nodes:
    abstract = node.abstract
    abstract_counts[abstract] = abstract_counts.get(abstract, 0) + 1

print("=== CWE Forest Statistics ===")
print(f"Total nodes: {len(all_nodes)}")
print(f"Total edges: {len(all_edges)}")
print(f"Root nodes: {len(roots)}")

print(f"\nNodes by abstract type:")
for abstract, count in sorted(abstract_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {abstract}: {count}")

=== CWE Forest Statistics ===
Total nodes: 938
Total edges: 928
Root nodes: 10

Nodes by abstract type:
  Base: 519
  Variant: 292
  Class: 110
  Pillar: 10
  Compound: 7


## Summary

The `cwe_tree` module provides a comprehensive API for:

1. **Node Access:** Get individual nodes by ID with automatic normalization
2. **Relationship Navigation:** Find parents, children, ancestors, and descendants
3. **Graph Traversal:** Use cpg2py's underlying traversal methods (succ, prev, descendants, ancestors)
4. **Forest Exploration:** Identify root nodes and overall structure
5. **Metadata Retrieval:** Access comprehensive node information and layer data
6. **Visualization:** Display tree structures with ASCII formatting

### Key Design Principles

- **Type Safety:** All methods return properly typed CweNode/CweEdge objects
- **ID Normalization:** CWE IDs are automatically normalized (e.g., "284" → "CWE-284")
- **Lazy Evaluation:** Traversal methods return iterables for memory efficiency
- **Error Handling:** Non-existent nodes return None gracefully
- **Forest Support:** Handles multiple independent root trees

For more details, see:
- `docs/design.md` - Architectural design and concepts
- `docs/traversal.md` - Detailed traversal API reference